In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct


In [23]:
import os
import json 
import pandas as pd
import uuid

In [28]:
jsonl_file_path = "../../data/vector_embeddings"

In [2]:
# connect client with server
client = QdrantClient(host = "localhost",port=6333 )

In [5]:
collection_name = "collections"
if client.collection_exists(collection_name):
    print("collection exist")
else:
    print("not exist")

not exist


In [11]:
client.recreate_collection(
    collection_name="demo_collection",
    vectors_config=VectorParams(
        size=4,
        distance= Distance.COSINE
        
    )
)

/var/folders/zj/fh53wsrn5kvgvytd_tv2sy3h0000gn/T/ipykernel_16445/3889230794.py:1: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [13]:
client.upsert(
    collection_name="demo_collection",
    points=[
        PointStruct(
            id=1,                         
            vector=[0.01, 0.45, -0.2, 0.5],            
            payload={                    
                "title": "Introduction to AI",
                "category": "education",
                "author": "John Doe"
            }
        ),
        PointStruct(
            id=2,
            vector=[0.01, 0.45, -0.12, 0.6],
            payload={
                "title": "Advanced Vector Databases",
                "category": "tech"
            }
        )
    ]
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [14]:
client.delete_collection(collection_name="demo_collection")

True

In [16]:
##sample collections
if(client.collection_exists("sample_docs_collections") != True):
    client.create_collection(
        collection_name="sample_docs_collections",
        vectors_config=VectorParams(
            size=768,
            distance=Distance.COSINE
        )  
    )

In [24]:
# change to points reading json file
all_points = []
for file_name in os.listdir(jsonl_file_path):
    if file_name.endswith(".jsonl"):
        file_path = os.path.join(jsonl_file_path, file_name)
        print(f"reading file name {file_name}")
        
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                data_item = json.loads(line)
                chunk_id = data_item['chunk_id']
                vector_data = data_item['vector']
                valid_uuid = str(uuid.uuid5(uuid.NAMESPACE_DNS, chunk_id))
                
                point = PointStruct(
                    id=valid_uuid,
                    vector=vector_data
                )
                all_points.append(point)
                
                

reading file name 28data_vector.jsonl
reading file name 19data_vector.jsonl
reading file name 22data_vector.jsonl
reading file name 29data_vector.jsonl
reading file name 35data_vector.jsonl
reading file name 21data_vector.jsonl


In [25]:
client.upsert(
    collection_name="sample_docs_collections",
    points = all_points
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [29]:
if(client.collection_exists("docs_collections") != True):
    client.create_collection(
        collection_name="docs_collections",
        vectors_config=VectorParams(
            size=768,
            distance=Distance.COSINE
        )  
    )

In [30]:
# on original data as size more so usert in batches
all_points_batch = []
qdrant_batch_size = 100
for file_name in os.listdir(jsonl_file_path):
    if file_name.endswith(".jsonl"):
        file_path = os.path.join(jsonl_file_path, file_name)
        print(f"reading file name {file_name}")
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                data_item = json.loads(line)
                chunk_id = data_item['chunk_id']
                vector_data = data_item['vector']
                valid_uuid = str(uuid.uuid5(uuid.NAMESPACE_DNS, chunk_id))
                
                point = PointStruct(
                    id=valid_uuid,
                    vector=vector_data
                )
                all_points_batch.append(point)
                if len(all_points_batch) == qdrant_batch_size:
                    client.upsert(
                        collection_name="docs_collections",
                        points = all_points_batch
                    )
                    all_points_batch = []

if all_points_batch: 
    client.upsert(
        collection_name="docs_collections",
        points = all_points_batch
    )
                    
print("All files processed and loaded successfully!")

reading file name 67data_vector.jsonl
reading file name 81data_vector.jsonl
reading file name 78data_vector.jsonl
reading file name 52data_vector.jsonl
reading file name 95data_vector.jsonl
reading file name 73data_vector.jsonl
reading file name 28data_vector.jsonl
reading file name 91data_vector.jsonl
reading file name 77data_vector.jsonl
reading file name 49data_vector.jsonl
reading file name 6data_vector.jsonl
reading file name 68data_vector.jsonl
reading file name 19data_vector.jsonl
reading file name 42data_vector.jsonl
reading file name 85data_vector.jsonl
reading file name 38data_vector.jsonl
reading file name 47data_vector.jsonl
reading file name 79data_vector.jsonl
reading file name 8data_vector.jsonl
reading file name 22data_vector.jsonl
reading file name 80data_vector.jsonl
reading file name 58data_vector.jsonl
reading file name 17data_vector.jsonl
reading file name 29data_vector.jsonl
reading file name 94data_vector.jsonl
reading file name 3data_vector.jsonl
reading file na